# This file contains Processes Exercises of Lecture 4

Since the Manning Basin in Australia has a subtropical to temperate climate, the snow exercise was not applicable for this area, so we did the exercise for the Rhine basin, just like the example.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import ewatercycle.forcing
import ewatercycle.observation.grdc
import ewatercycle.analysis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
from pathlib import Path
from cartopy.io import shapereader
from rich import print
import matplotlib.pyplot as plt
import xarray as xr
import shutil


In [4]:
own_region = 'Rhine'  # for example: "Rhine"

# Shapefile that describes the basin we want to study
path = Path.cwd()
forcing_path = path / "Forcing"
shapeFile = forcing_path / f"{own_region}.shp"
# Location where forcing from other notebook was stored
forcingLocation = forcing_path / f"{own_region}_Forcing"

In [5]:
# Period of interest
experiment_start_time = "2010-10-01T00:00:00Z"
experiment_end_time = "2011-09-30T00:00:00Z"

ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].generate(
    dataset="ERA5",
    start_time=experiment_start_time,
    end_time=experiment_end_time,
    shape=shapeFile.absolute(),
)


In [9]:

# Name of shapefile/region
own_region = "Rhine"

# Shapefile that describes the basin we want to study
path = Path.cwd()
forcing_path = path / "Forcing"
shapeFile = forcing_path / f"{own_region}.shp"

# Location to saved forcing results from previous notebook
forcingLocation = forcing_path / f"{own_region}_2010_2011_Forcing"


# ERA5 data Rhine
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].generate(
    dataset="ERA5",
    start_time=experiment_start_time,
    end_time=experiment_end_time,
    shape=shapeFile.absolute(),
)


ERROR:esmvalcore._recipe.recipe:Could not create all tasks
ERROR:esmvalcore._recipe.recipe:In preprocessor function `extract_shape`: Unable to find 'shapefile: /home/group10/teaching-materials/book/exercise_journal_ann_laura/Forcing/Rhine.shp'
ERROR:esmvalcore._recipe.recipe:In preprocessor function `extract_shape`: Unable to find 'shapefile: /home/group10/teaching-materials/book/exercise_journal_ann_laura/Forcing/Rhine.shp'
ERROR:esmvalcore._recipe.recipe:In preprocessor function `extract_shape`: Unable to find 'shapefile: /home/group10/teaching-materials/book/exercise_journal_ann_laura/Forcing/Rhine.shp'


RecipeError: Could not create all tasks

In [10]:
temp = ERA5_forcing_Rhine.to_xarray()["tas"]
prec = ERA5_forcing_Rhine.to_xarray()["pr"]
temp
prec = prec * 86400
temp = temp - 272.15
temp["time"] = temp["time"].dt.date
prec["time"] = prec["time"].dt.date

NameError: name 'ERA5_forcing_Rhine' is not defined

In [ ]:
temp.name = "T C" 
prec.name = "P mm/d"

df = xr.merge([temp, prec]).to_dataframe()
Rhine = df[["T C", "P mm/d"]].dropna()
Rhine.index = pd.to_datetime(Rhine.index)
Rhine = Rhine.loc["2010-10-01":"2011-09-30"]
Rhine

In [ ]:
# Test with exercise data

data = pd.read_excel('Snow1.xlsx', skiprows=(0, 1, 3), usecols=[0, 1, 2, 3, 4], index_col=[0], parse_dates=True)
data['P mm/d'] = data['Precipitation']
data['T C'] = data['Temperature TR']

Case 1: Flat terrain

Tt = -0.5
Fm = 3
Er = 360 # average elevation Rhine area
dt = 1

def snow_melt(df, Tt, Fm, dt):
    t = np.arange(len(df['P mm/d']))
    Pr = np.zeros(len(t))
    Ps = np.zeros(len(t))
    M = np.zeros(len(t))
    Ssnow = np.zeros(len(t))
    Pl = np.zeros(len(t))

    for i in np.arange(1, len(t)):
        if df['T C'][i] <= Tt:
            Pr[i] = 0
            Ps[i] = df['P mm/d'][i]
            M[i] = 0
            Ssnow[i] = Ssnow[i - 1] + Ps[i] * dt
            Pl[i] = 0

        else:
            Pr[i] = df['P mm/d'][i]
            Ps[i] = 0
            M[i] = np.minimum(Ssnow[i-1] / dt, Fm * (df['T C'][i] - Tt))
            Ssnow[i] = Ssnow[i - 1] - M[i] * dt
            Pl[i] = Pr[i] + M[i]
    
    return t, Pr, Ps, M, Ssnow, Pl

In [ ]:
t, Pr, Ps, M, Ssnow, Pl = snow_melt(Rhine, Tt, Fm, dt)

plt.plot(Rhine.index, Ssnow, label="Snow Storage")
plt.title('Snow storage Rhine 2010-2011 (flat terrain)')
plt.xlabel('Time')
plt.ylabel('SWE [mm]')
plt.legend()

In [ ]:
plt.plot(Rhine.index, Pl, label='Pl')
plt.title('Liquid water input to the soil Rhine 2010-2011 (flat terrain)')
plt.xlabel('Time')
plt.ylabel('Pl [mm/d]')
plt.legend()

In [ ]:
# Test with exercise data
t, Pr, Ps, M, Ssnow, Pl = snow_melt(data, Tt, Fm, dt)

plt.plot(t, Ssnow)
plt.plot(t, Pl)

Case 2: Difference in elevation band

shapeObject = shapereader.Reader(shapeFile.absolute())
record = next(shapeObject.records())
shape_area = record.attributes["SUB_AREA"] * 1e6
print("The catchment area is:", shape_area)

In [ ]:
# Estimation elevation bands catchment of the Rhine
E1 = 300
E2 = 800
E3 = 1300
E4 = 1800

A1 = 0.25 * shape_area
A2 = 0.25 * shape_area
A3 = 0.25 * shape_area
A4 = 0.25 * shape_area

In [ ]:
def snow_melt(df, Tt, Fm, dt, Ei, Er):
    Ti = df['T C'] - 0.6 * (Ei - Er) / 100
    t = np.arange(len(df['P mm/d']))
    Pr = np.zeros(len(t))
    Ps = np.zeros(len(t))
    M = np.zeros(len(t))
    Ssnow = np.zeros(len(t))
    Pl = np.zeros(len(t))

    for i in np.arange(1, len(t)):
        if Ti[i] <= Tt:
            Pr[i] = 0
            Ps[i] = df['P mm/d'][i]
            M[i] = 0
            Ssnow[i] = Ssnow[i - 1] + Ps[i] * dt
            Pl[i] = 0

        else:
            Pr[i] = df['P mm/d'][i]
            Ps[i] = 0
            M[i] = np.minimum(Ssnow[i-1] / dt, Fm * (Ti[i] - Tt))
            Ssnow[i] = Ssnow[i - 1] - M[i] * dt
            Pl[i] = Pr[i] + M[i]

        # print(df['T C'][i], Ei, Er, Ti[i], Pr[i], Ps[i], M[i], Ssnow[i])
    
    return t, Pr, Ps, M, Ssnow, Pl

In [ ]:
_, _, _, _, Ssnow1, Pl1 = snow_melt(Rhine, Tt, Fm, dt, E1, Er)
_, _, _, _, Ssnow2, Pl2 = snow_melt(Rhine, Tt, Fm, dt, E2, Er)
_, _, _, _, Ssnow3, Pl3 = snow_melt(Rhine, Tt, Fm, dt, E3, Er)
_, _, _, _, Ssnow4, Pl4 = snow_melt(Rhine, Tt, Fm, dt, E4, Er)

In [ ]:
A_tot = shape_area
Pl_tot = A1 / A_tot * Pl1 + A2 / A_tot * Pl2 + A3 / A_tot * Pl3 + A4 / A_tot * Pl4
Ssnow_tot = A1 / A_tot * Ssnow1 + A2 / A_tot * Ssnow2 + A3 / A_tot * Ssnow3 + A4 / A_tot * Ssnow4
plt.plot(Rhine.index, Pl_tot, label='Pl_tot')
plt.title('Liquid water input to the soil Rhine 2010-2011 (elevated terrain)')
plt.xlabel('Time')
plt.ylabel('Pl_tot [mm/d]')
plt.legend()

In [ ]:
plt.plot(Rhine.index, Ssnow_tot, label='Ssnow_tot')
plt.title('Snow storage Rhine 2010-2011 (elevated terrain)')
plt.xlabel('Time')
plt.ylabel('SWE [mm]')
plt.legend()

Case 3: Increased temperature by 3 degrees

Rhine['T+2 C'] = Rhine['T C'] + 3

In [ ]:
def snow_melt(df, Tt, Fm, dt, Ei, Er):
    Ti = df['T+2 C'] - 0.6 * (Ei - Er) / 100
    t = np.arange(len(df['P mm/d']))
    Pr = np.zeros(len(t))
    Ps = np.zeros(len(t))
    M = np.zeros(len(t))
    Ssnow = np.zeros(len(t))
    Pl = np.zeros(len(t))

    for i in np.arange(1, len(t)):
        if Ti[i] <= Tt:
            Pr[i] = 0
            Ps[i] = df['P mm/d'][i]
            M[i] = 0
            Ssnow[i] = Ssnow[i - 1] + Ps[i] * dt
            Pl[i] = 0

        else:
            Pr[i] = df['P mm/d'][i]
            Ps[i] = 0
            M[i] = np.minimum(Ssnow[i-1] / dt, Fm * (Ti[i] - Tt))
            Ssnow[i] = Ssnow[i - 1] - M[i] * dt
            Pl[i] = Pr[i] + M[i]

        # print(df['T C'][i], Ei, Er, Ti[i], Pr[i], Ps[i], M[i], Ssnow[i])
    
    return t, Pr, Ps, M, Ssnow, Pl

In [ ]:
_, _, _, _, Ssnow31, Pl31 = snow_melt(Rhine, Tt, Fm, dt, E1, Er)
_, _, _, _, Ssnow32, Pl32 = snow_melt(Rhine, Tt, Fm, dt, E2, Er)
_, _, _, _, Ssnow33, Pl33 = snow_melt(Rhine, Tt, Fm, dt, E3, Er)
_, _, _, _, Ssnow34, Pl34 = snow_melt(Rhine, Tt, Fm, dt, E4, Er)

In [ ]:
Pl_tot3 = A1 / A_tot * Pl31 + A2 / A_tot * Pl32 + A3 / A_tot * Pl33 + A4 / A_tot * Pl34
Ssnow_tot3 = A1 / A_tot * Ssnow31 + A2 / A_tot * Ssnow32 + A3 / A_tot * Ssnow33 + A4 / A_tot * Ssnow34

plt.figure(figsize=(10, 4))

plt.plot(Rhine.index, Pl, label='case 1')
plt.plot(Rhine.index, Pl_tot, label='case 2')
plt.plot(Rhine.index, Pl_tot3, label='case 3')
plt.title('Liquid water input to the soil Rhine 2010-2011')
plt.xlabel('Time')
plt.ylabel('Pl [mm/d]')
plt.legend()


plt.figure(figsize=(10, 4))

plt.plot(Rhine.index, Ssnow_tot3, label='case 3')
plt.plot(Rhine.index, Ssnow_tot, label='case 2')
plt.plot(Rhine.index, Ssnow, label='case 1')

plt.title('Snow storage Rhine 2010-2011')
plt.xlabel('Time')
plt.ylabel('SWE [mm]')
plt.legend()